# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/estuchis21/ml-internship-week1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one content item corresponding to a particular client on a particular date

Time window:
March 2026 (month=2026-03)

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
!pip install -q datasets huggingface_hub pandas pyarrow
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))

from huggingface_hub import list_repo_files

files = list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset"
)

import pandas as pd

url = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"

df = pd.read_parquet(url)

df.info()
df.head()



## 2. Fields: feature / label / context / excluded

Features

The features used to identify content refresh opportunities are:

gsc_impressions: measures how many times the content appeared in Google Search results. It represents search visibility available before the decision moment.
gsc_clicks: measures the number of clicks generated from search results. It represents observed search traffic performance.
gsc_avg_position: represents the average ranking position of the content in search results. It indicates the content's search placement.
ga4_sessions: measures the number of user sessions reaching the content. It provides historical traffic information.
scroll_events: measures user interaction with the content and provides an engagement signal.
Label

refresh_opportunity

A proxy label indicating whether a content item could represent a refresh opportunity.

The proxy is created from observed performance signals:

content receives search visibility (gsc_impressions)
but does not generate clicks (gsc_clicks)

This represents content that is visible in search but may have optimization potential.

Context

The following fields provide identification and temporal context:

report_date: date when the performance observation was recorded.
month: dataset partition month.
client_hash_id: identifier of the client associated with the content.
content_hash_id: identifier of the content item.
Excluded

The following fields are excluded from model features:

client_hash_id: excluded because it is only an identifier. Using it as a feature could make the model memorize specific clients instead of learning general patterns.
content_hash_id: excluded because it identifies individual content items and does not represent a measurable performance signal.
Future performance information: excluded because it would not be available at the moment of making a refresh decision and would introduce data leakage.

In [ ]:

# 2. Imputar posición
df["gsc_avg_position"] = df["gsc_avg_position"].fillna(
    df["gsc_avg_position"].mean()
)


# 3. Imputar GA4
ga4_features = [
    "ga4_sessions",
    "ga4_engaged_sessions",
    "sessions_ai",
    "scroll_events"
]

df[ga4_features] = df[ga4_features].fillna(0)


features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "scroll_events"
]

# 5. Dataset de entrada
X = df[features]

X

df["refresh_opportunity"] = (
    (df["gsc_impressions"] > 15) &
    (df["gsc_clicks"] == 0)
).astype(int)


target = df['refresh_opportunity']

df_corr = X.copy()

df_corr["refresh_opportunity"] = df["refresh_opportunity"]

df["refresh_opportunity"].value_counts()

correlation = df_corr.corr(
    numeric_only=True
)

correlation


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
duplicated = df.duplicated(
    subset=["report_date", "client_hash_id", "content_hash_id"]
).sum()

print("Duplicates:", duplicated)

print(df.shape[0])
print(df['report_date'].max())
print(df['report_date'].min())


gsc_available = df[
    df["gsc_data_available"] == True
].shape[0]


ga4_available = df[
    df["ga4_data_available"] == True
].shape[0]


print("GSC available:", gsc_available)
print("GA4 available:", ga4_available)



## Data observations and limitations

### Client distribution

The dataset contains observations from multiple clients, but the number of records is not evenly distributed. Some clients contribute a very large number of rows, while others have significantly fewer observations. This indicates that the historical data is concentrated around a subset of clients, so model performance may be influenced by clients with more available records.

### Search visibility distribution (GSC)

Search visibility is highly concentrated. Most content observations have zero impressions (`gsc_impressions = 0`), with 6,230,317 records having no observed impressions or clicks. Only a smaller portion of content receives search exposure, and a few content items account for very high impression values.

This means that zero impressions should not automatically be interpreted as poor content quality. It may indicate lack of search exposure, indexing limitations, or low demand.

### Click behavior

Clicks are also highly sparse. Many observations have impressions but no clicks, meaning that some content is visible in search results but does not generate user visits. These cases are more relevant for identifying refresh opportunities because the content already has search exposure.

### GA4 availability and engagement

GA4 sessions are mostly concentrated around zero values. The majority of observations have no recorded GA4 sessions, while only a smaller number of records show user engagement. This indicates that analytics availability and traffic levels vary significantly across observations.

### Target distribution

The proxy target `needs_refreshing` is imbalanced:

- Class 0: 8,163,936 observations
- Class 1: 1,677,442 observations

Most observations are classified as not needing refresh. Therefore, accuracy alone would not be a reliable evaluation metric, and metrics focused on identifying positive cases (such as recall) are more appropriate.

### Limitations

This dataset provides observed performance signals, but it cannot explain the reason behind user behavior. For example, low clicks may be caused by poor titles, search intent mismatch, competition, or other external factors that are not measured in the dataset.

Additionally, daily observations may contain repeated measurements of the same content over time, meaning that consecutive rows may not be completely independent.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print(df['client_hash_id'].value_counts().head(60))
df.columns


print(df['report_date'].value_counts())
print(df['refresh_opportunity'].value_counts())


print(df['gsc_impressions'].value_counts())
print(df['ga4_sessions'].value_counts())


print(df.groupby("gsc_impressions")["gsc_clicks"].sum())

print(df.groupby("gsc_impressions")["gsc_clicks"].size())
#

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.